# Full session after the first Ensure delivery

Cache lifetime, licking, and every Ensure delivery from 60 seconds before the first Ensure through the end of each recording. The slow stage runs once. Later cells can crop the cached data to any shorter view without reopening `.doric` files.

In [ ]:
from datetime import date, datetime
from pathlib import Path
from time import perf_counter
import csv
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
from fluopulse_analysis import find_doric_files, read_doric

## Configuration

Leave `POST_WINDOW_S = None` to retain everything after the first Ensure until each recording ends. A numeric value limits the slow cache to that many seconds.

In [ ]:
SESSION_TABLE = Path.home() / "Documents" / "SC_FLiPmice_2025.xlsx"
DORIC_ROOT = Path(r"Z:\FLIM FLIP")
OUTPUT_FOLDER = Path.cwd() / "analysis_output_full_ensure_session"
CACHE_FILE = OUTPUT_FOLDER / "full_ensure_session_cache.npz"
PRE_WINDOW_S = 60.0
POST_WINDOW_S = None  # None means through the end of every recording
BASELINE_WINDOW_S = (-30.0, -5.0)
SAMPLE_INTERVAL_S = 0.1
MIN_BASELINE_COVERAGE = 0.8
print(f"Doric root available: {DORIC_ROOT.is_dir()} ({DORIC_ROOT})")

In [ ]:
def normalize_date(value):
    if isinstance(value, (date, datetime)):
        return value.strftime("%y%m%d")
    return str(int(value)).zfill(6)

workbook = load_workbook(SESSION_TABLE, data_only=True, read_only=True)
worksheet = workbook.active
rows = list(worksheet.iter_rows(values_only=True))
workbook.close()
headers = [str(value).strip() if value is not None else "" for value in rows[0]]
if not headers or headers[0].casefold() != "mouse":
    raise ValueError("The first workbook column must be named 'mouse'")
session_records = []
for row in rows[1:]:
    if not row or row[0] is None:
        continue
    mouse = str(row[0]).strip()
    for column, value in zip(headers[1:], row[1:], strict=False):
        if value is not None:
            session_records.append({"mouse": mouse, "date": normalize_date(value), "source_column": column})
print(f"Loaded {len(session_records)} mouse-date sessions")

## Slow processing and long-cache creation

Run this once. It reads only fitted lifetime and digital events; raw waveform matrices remain lazy and are not loaded.

In [ ]:
def recording_label(path, mouse, session_date):
    stem = path.stem
    for prefix in (f"{mouse}_{session_date}_", f"{session_date}_{mouse}_"):
        if stem.casefold().startswith(prefix.casefold()):
            return stem[len(prefix):]
    return stem

def write_dict_rows(path, rows):
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

qc_rows = []
recordings = []
processing_started = perf_counter()
total_sessions = len(session_records)
print(f"Starting {total_sessions} mouse-date sessions", flush=True)
for session_index, record in enumerate(session_records, start=1):
    mouse = record["mouse"]
    session_date = record["date"]
    print(f"[{session_index}/{total_sessions}] {mouse} {session_date}: searching...", flush=True)
    doric_files = find_doric_files(mouse, session_date, doric_root=DORIC_ROOT)
    if not doric_files:
        qc_rows.append({"mouse": mouse, "date": session_date, "recording": "", "status": "missing .doric"})
        print("    no matching .doric files", flush=True)
        continue
    for file_index, doric_file in enumerate(doric_files, start=1):
        label = recording_label(doric_file, mouse, session_date)
        file_started = perf_counter()
        print(f"    ({file_index}/{len(doric_files)}) {label}: reading...", flush=True)
        try:
            recording = read_doric(doric_file)
            ensure_pulses = recording.digital_pulses.get("ensure")
            lick_pulses = recording.digital_pulses.get("licking")
            ensure_times = np.array([]) if ensure_pulses is None else ensure_pulses.onset_times
            lick_times = np.array([]) if lick_pulses is None else lick_pulses.onset_times
            if ensure_times.size == 0:
                qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": "no Ensure events"})
                print("        skipped: no Ensure events", flush=True)
                continue
            first_ensure_s = float(ensure_times[0])
            relative_lifetime_time = recording.time - first_ensure_s
            available_start_s = float(relative_lifetime_time[0])
            available_end_s = float(relative_lifetime_time[-1])
            baseline_mask = (relative_lifetime_time >= BASELINE_WINDOW_S[0]) & (relative_lifetime_time < BASELINE_WINDOW_S[1])
            baseline_coverage = float(np.mean(np.isfinite(recording.tau_ns[baseline_mask]))) if np.any(baseline_mask) else 0.0
            if baseline_coverage == 0:
                raise ValueError("No finite lifetime samples in the baseline window")
            baseline_tau = float(np.nanmean(recording.tau_ns[baseline_mask]))
            status = "ok" if baseline_coverage >= MIN_BASELINE_COVERAGE else "limited baseline coverage"
            recordings.append({
                "mouse": mouse, "date": session_date, "recording": label,
                "source_file": str(doric_file), "first_ensure_s": first_ensure_s,
                "relative_lifetime_time": relative_lifetime_time, "tau_ns": recording.tau_ns,
                "lick_relative_s": lick_times - first_ensure_s,
                "ensure_relative_s": ensure_times - first_ensure_s,
                "baseline_tau_ns": baseline_tau, "baseline_coverage": baseline_coverage,
                "available_start_s": available_start_s, "available_end_s": available_end_s,
            })
            qc_rows.append({
                "mouse": mouse, "date": session_date, "recording": label, "status": status,
                "first_ensure_s": first_ensure_s, "baseline_coverage": baseline_coverage,
                "available_post_s": available_end_s, "ensure_count": int(ensure_times.size),
            })
            print(f"        first Ensure {first_ensure_s:.2f} s; retained through +{available_end_s:.1f} s ({perf_counter() - file_started:.1f} s)", flush=True)
        except Exception as error:
            qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": f"error: {error}"})
            print(f"        ERROR after {perf_counter() - file_started:.1f} s: {error}", flush=True)
if not recordings:
    raise ValueError("No recordings contained a usable Ensure event")
maximum_available_post_s = max(item["available_end_s"] for item in recordings)
cache_post_s = maximum_available_post_s if POST_WINDOW_S is None else min(float(POST_WINDOW_S), maximum_available_post_s)
relative_time = np.arange(-PRE_WINDOW_S, cache_post_s + SAMPLE_INTERVAL_S / 2, SAMPLE_INTERVAL_S)
relative_time[np.isclose(relative_time, 0.0, atol=SAMPLE_INTERVAL_S / 1000)] = 0.0
bin_edges = np.concatenate([relative_time - SAMPLE_INTERVAL_S / 2, [relative_time[-1] + SAMPLE_INTERVAL_S / 2]])
tau_rows = []
delta_tau_rows = []
lick_rows = []
ensure_rows = []
for item in recordings:
    tau_trace = np.interp(relative_time, item["relative_lifetime_time"], item["tau_ns"], left=np.nan, right=np.nan)
    tau_rows.append(tau_trace)
    delta_tau_rows.append(tau_trace - item["baseline_tau_ns"])
    available = (relative_time >= item["available_start_s"]) & (relative_time <= item["available_end_s"])
    lick_trace = np.histogram(item["lick_relative_s"], bins=bin_edges)[0].astype(float)
    ensure_trace = np.histogram(item["ensure_relative_s"], bins=bin_edges)[0].astype(float)
    lick_trace[~available] = np.nan
    ensure_trace[~available] = np.nan
    lick_rows.append(lick_trace)
    ensure_rows.append(ensure_trace)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    CACHE_FILE, relative_time_s=relative_time, tau_ns=np.vstack(tau_rows), delta_tau_ns=np.vstack(delta_tau_rows),
    lick_counts_per_bin=np.vstack(lick_rows), ensure_counts_per_bin=np.vstack(ensure_rows),
    bin_width_s=np.asarray(SAMPLE_INTERVAL_S),
    mouse=np.asarray([item["mouse"] for item in recordings], dtype=str),
    session_date=np.asarray([item["date"] for item in recordings], dtype=str),
    recording=np.asarray([item["recording"] for item in recordings], dtype=str),
    source_file=np.asarray([item["source_file"] for item in recordings], dtype=str),
    first_ensure_s=np.asarray([item["first_ensure_s"] for item in recordings], dtype=float),
    baseline_tau_ns=np.asarray([item["baseline_tau_ns"] for item in recordings], dtype=float),
    baseline_coverage=np.asarray([item["baseline_coverage"] for item in recordings], dtype=float),
    available_start_s=np.asarray([item["available_start_s"] for item in recordings], dtype=float),
    available_end_s=np.asarray([item["available_end_s"] for item in recordings], dtype=float),
)
write_dict_rows(OUTPUT_FOLDER / "recording_qc.csv", qc_rows)
print(f"Finished in {perf_counter() - processing_started:.1f} s", flush=True)
print(f"Cached {len(recordings)} recordings from -{PRE_WINDOW_S:g} to +{cache_post_s:.1f} s", flush=True)
print(CACHE_FILE.resolve())
qc_rows

## Fast cached views

Change `VIEW_WINDOW_S` freely after the slow pass. For example, use `(-30, 120)`, `(-60, 300)`, or `(-60, None)` for the entire cached session.

In [ ]:
MOUSE_FILTER = []  # Example: ["SC43", "SC44"]
DATE_FILTER = []  # Example: ["250625"]
RECORDING_TEXT = ""  # Example: "FoodReward"
VIEW_WINDOW_S = (-60.0, None)  # None uses the complete cached post-Ensure period
SHOW_LICKING = True
SHOW_ENSURE_SCHEDULE = True
HEATMAP_DISPLAY_BIN_S = 1.0

if not CACHE_FILE.is_file():
    raise FileNotFoundError(f"Run the slow processing cell once to create {CACHE_FILE}")
with np.load(CACHE_FILE, allow_pickle=False) as cache:
    cached_time = cache["relative_time_s"]
    all_tau = cache["tau_ns"]
    all_delta_tau = cache["delta_tau_ns"]
    all_lick_counts = cache["lick_counts_per_bin"].astype(float)
    all_ensure_counts = cache["ensure_counts_per_bin"].astype(float)
    bin_width_s = float(cache["bin_width_s"])
    all_mice = cache["mouse"].astype(str)
    all_dates = cache["session_date"].astype(str)
    all_recordings = cache["recording"].astype(str)
    all_available_end = cache["available_end_s"].astype(float)
selected_recordings = np.ones(all_delta_tau.shape[0], dtype=bool)
if MOUSE_FILTER:
    selected_recordings &= np.isin(all_mice, np.asarray(MOUSE_FILTER, dtype=str))
if DATE_FILTER:
    selected_recordings &= np.isin(all_dates, np.asarray(DATE_FILTER, dtype=str))
if RECORDING_TEXT:
    selected_recordings &= np.char.find(np.char.lower(all_recordings), RECORDING_TEXT.casefold()) >= 0
view_start = float(VIEW_WINDOW_S[0])
view_end = cached_time[-1] if VIEW_WINDOW_S[1] is None else float(VIEW_WINDOW_S[1])
selected_time = (cached_time >= view_start) & (cached_time <= view_end)
if not np.any(selected_recordings) or not np.any(selected_time):
    raise ValueError("No cached data match the current recording and time filters")
relative_time = cached_time[selected_time]
tau = all_tau[selected_recordings][:, selected_time]
delta_tau = all_delta_tau[selected_recordings][:, selected_time]
lick_counts = all_lick_counts[selected_recordings][:, selected_time]
ensure_counts = all_ensure_counts[selected_recordings][:, selected_time]
recording_mice = all_mice[selected_recordings]
recording_dates = all_dates[selected_recordings]
recording_labels = all_recordings[selected_recordings]
available_end_s = all_available_end[selected_recordings]
print(f"Selected {delta_tau.shape[0]} recordings and {relative_time[0]:.1f} to {relative_time[-1]:.1f} s")
print(f"Mice: {', '.join(np.unique(recording_mice))}")

In [ ]:
mouse_names = np.unique(recording_mice)
block_size = max(1, int(round(HEATMAP_DISPLAY_BIN_S / bin_width_s)))
block_slices = [slice(start, min(start + block_size, relative_time.size)) for start in range(0, relative_time.size, block_size)]
heatmap_time = np.asarray([np.mean(relative_time[section]) for section in block_slices])
def block_mean(values):
    reduced = []
    for section in block_slices:
        block = values[:, section]
        count = np.sum(np.isfinite(block), axis=1)
        total = np.nansum(block, axis=1)
        reduced.append(np.divide(total, count, out=np.full(total.shape, np.nan), where=count > 0))
    return np.column_stack(reduced)
def block_any(values):
    reduced = []
    for section in block_slices:
        block = values[:, section]
        present = np.any(block > 0, axis=1).astype(float)
        present[np.all(~np.isfinite(block), axis=1)] = np.nan
        reduced.append(present)
    return np.column_stack(reduced)
def finite_mean(values):
    count = np.sum(np.isfinite(values), axis=0)
    total = np.nansum(values, axis=0)
    return np.divide(total, count, out=np.full(total.shape, np.nan), where=count > 0)
def finite_sem(values):
    count = np.sum(np.isfinite(values), axis=0)
    mean = finite_mean(values)
    squared_error = np.nansum((values - mean) ** 2, axis=0)
    sem = np.zeros_like(mean)
    valid = count > 1
    sem[valid] = np.sqrt(squared_error[valid] / (count[valid] - 1)) / np.sqrt(count[valid])
    sem[count == 0] = np.nan
    return sem
lifetime_heatmap = block_mean(delta_tau)
lick_heatmap = block_any(lick_counts)
ensure_heatmap = block_any(ensure_counts)
mouse_lifetime_means = np.vstack([finite_mean(delta_tau[recording_mice == mouse]) for mouse in mouse_names])
grand_lifetime_mean = finite_mean(mouse_lifetime_means)
lifetime_sem = finite_sem(mouse_lifetime_means)
panel_count = 2 + (2 if SHOW_LICKING else 0) + int(SHOW_ENSURE_SCHEDULE)
height_ratios = [1.2, 1.0] + ([1.0, 1.0] if SHOW_LICKING else []) + ([0.7] if SHOW_ENSURE_SCHEDULE else [])
figure, axes = plt.subplots(panel_count, 1, figsize=(14, 2.6 * panel_count), sharex=True, constrained_layout=True, gridspec_kw={"height_ratios": height_ratios})
axes = np.atleast_1d(axes)
figure.suptitle("Full session aligned to first Ensure delivery")
image = axes[0].imshow(lifetime_heatmap, aspect="auto", origin="lower", extent=[heatmap_time[0] - HEATMAP_DISPLAY_BIN_S / 2, heatmap_time[-1] + HEATMAP_DISPLAY_BIN_S / 2, 0, delta_tau.shape[0]], cmap="coolwarm", interpolation="none", resample=False)
axes[0].set_ylabel("Recording")
figure.colorbar(image, ax=axes[0], label="Delta tau (ns)")
for trace in mouse_lifetime_means:
    axes[1].plot(relative_time, trace, color="0.75", linewidth=0.8)
axes[1].plot(relative_time, grand_lifetime_mean, color="tab:blue", linewidth=2, label=f"Mouse mean (n={mouse_names.size})")
axes[1].fill_between(relative_time, grand_lifetime_mean - lifetime_sem, grand_lifetime_mean + lifetime_sem, color="tab:blue", alpha=0.2)
axes[1].set_ylabel("Delta tau (ns)")
axes[1].legend()
next_axis = 2
if SHOW_LICKING:
    axes[next_axis].imshow(lick_heatmap, aspect="auto", origin="lower", extent=[heatmap_time[0] - HEATMAP_DISPLAY_BIN_S / 2, heatmap_time[-1] + HEATMAP_DISPLAY_BIN_S / 2, 0, lick_counts.shape[0]], cmap="Greys", vmin=0, vmax=1, interpolation="none", resample=False)
    axes[next_axis].set_ylabel("Lick raster")
    next_axis += 1
    lick_rates = lick_counts / bin_width_s
    mouse_lick_means = np.vstack([finite_mean(lick_rates[recording_mice == mouse]) for mouse in mouse_names])
    grand_lick_mean = finite_mean(mouse_lick_means)
    lick_sem = finite_sem(mouse_lick_means)
    for trace in mouse_lick_means:
        axes[next_axis].plot(relative_time, trace, color="0.75", linewidth=0.8)
    axes[next_axis].plot(relative_time, grand_lick_mean, color="tab:green", linewidth=2, label=f"Mouse mean (n={mouse_names.size})")
    axes[next_axis].fill_between(relative_time, grand_lick_mean - lick_sem, grand_lick_mean + lick_sem, color="tab:green", alpha=0.2)
    axes[next_axis].set_ylabel("Lick rate (Hz)")
    axes[next_axis].legend()
    next_axis += 1
if SHOW_ENSURE_SCHEDULE:
    axes[next_axis].imshow(ensure_heatmap, aspect="auto", origin="lower", extent=[heatmap_time[0] - HEATMAP_DISPLAY_BIN_S / 2, heatmap_time[-1] + HEATMAP_DISPLAY_BIN_S / 2, 0, ensure_counts.shape[0]], cmap="Oranges", vmin=0, vmax=1, interpolation="none", resample=False)
    axes[next_axis].set_ylabel("Ensure raster")
for axis in axes:
    axis.axvline(0, color="tab:orange", linestyle="--")
axes[-1].set_xlabel("Time from first Ensure delivery (s)")
plt.show()

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUTPUT_FOLDER / "full_ensure_session_selected.npz", relative_time_s=relative_time, tau_ns=tau, delta_tau_ns=delta_tau, lick_counts_per_bin=lick_counts, ensure_counts_per_bin=ensure_counts, bin_width_s=np.asarray(bin_width_s), mouse=recording_mice, session_date=recording_dates, recording=recording_labels, available_end_s=available_end_s)
figure.savefig(OUTPUT_FOLDER / "full_ensure_session.png", dpi=200)
print(f"Wrote selected view to {OUTPUT_FOLDER.resolve()}")

Rows become missing after an individual recording ends; later group means therefore use only recordings and mice still contributing finite lifetime samples. The Ensure raster shows every delivery retained in the long cache.